In [1]:
import os

import pandas as pd
import numpy as np

# Odczyt datasetu

In [2]:
df = pd.read_csv(os.path.join("../data", "domy.csv"))

# Zamiana kilku danych tekstowych na numeryczne

In [3]:
df = df.copy()
df["LotFrontage"] = pd.to_numeric(df["LotFrontage"], errors="coerce")
df["MasVnrArea"] = pd.to_numeric(df["MasVnrArea"], errors="coerce")
df["GarageYrBlt"] = pd.to_numeric(df["GarageYrBlt"], errors="coerce")

# Usuwanie kolumn

In [4]:
str_cols = ["Utilities", "Street", "Condition2", "PoolQC", "RoofMatl"]
# num_cols = ["3SsnPorch", "LowQualFinSF", "PoolArea"]
low_corr_cols = ["YrSold", "MSSubClass", "BsmtFinSF2", "MoSold"]
# pair_corr_cols = ["YearBuilt", "TotRmsAbvGrd"]
df = df.copy().drop(str_cols, axis=1)
# df = df.copy().drop(num_cols, axis=1)
df = df.copy().drop(low_corr_cols, axis=1)
# df = df.copy().drop(pair_corr_cols, axis=1)

# Podział danych na zbiór treningowy i testowy

In [5]:
from sklearn.model_selection import train_test_split

X = df.drop("SalePrice", axis=1)
y = df["SalePrice"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# XGBoost

## Inicjalizacja modelu i reportera do treningów

In [6]:
from Regression.training_reporter import TrainingReporter
from Regression.training_model import CustomRegressionModel
from xgboost import XGBRegressor

model = CustomRegressionModel(XGBRegressor(
            n_estimators=1000,
            learning_rate=0.05,
            max_depth=3,
            min_child_weight=3,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.1,
            reg_lambda=0.1,
            objective='reg:squarederror',
            random_state=42,
            n_jobs=-1
            ))
reporter = TrainingReporter(model, X_train, X_test, y_train, y_test, 'XGB')

## Kroswalidacja

In [7]:
reporter.run_cross_validation(cv=5)

Start cross validation...
Iteration 1
Fold 0: RMSE = 22892.128952982945, MSE = 524049568.0, R^2 = 0.9189255833625793
Fold 1: RMSE = 33128.344842445724, MSE = 1097487232.0, R^2 = 0.8125017285346985
Fold 2: RMSE = 32679.88151753308, MSE = 1067974656.0, R^2 = 0.8414266109466553
Fold 3: RMSE = 22181.236034089714, MSE = 492007232.0, R^2 = 0.9098556041717529
Fold 4: RMSE = 20386.89382912463, MSE = 415625440.0, R^2 = 0.9212133884429932
Iteration 2
Fold 0: RMSE = 33773.49114320283, MSE = 1140648704.0, R^2 = 0.8276606202125549
Fold 1: RMSE = 23982.452251594288, MSE = 575158016.0, R^2 = 0.9099738597869873
Fold 2: RMSE = 21355.473677724876, MSE = 456056256.0, R^2 = 0.9082237482070923
Fold 3: RMSE = 27121.769853754013, MSE = 735590400.0, R^2 = 0.8517436981201172
Fold 4: RMSE = 26394.992252319378, MSE = 696695616.0, R^2 = 0.8967471718788147
Iteration 3
Fold 0: RMSE = 20226.022050813648, MSE = 409091968.0, R^2 = 0.8990859389305115
Fold 1: RMSE = 30673.75686152578, MSE = 940879360.0, R^2 = 0.85900473

## Grid search

In [8]:
params_grid = {
            'model__n_estimators': [500, 1000, 2000],
            'model__learning_rate': [0.01, 0.05, 0.1],
            'model__max_depth': [3, 4, 5],
            'model__min_child_weight': [1, 2, 3],
            'model__subsample': [0.6, 0.8],
            'model__colsample_bytree': [0.6, 0.8],
        }

reporter.run_grid_search(cv=5, param_grid=params_grid)

Start grid search...
Grid finished!
Best params: {'model__colsample_bytree': 0.6, 'model__learning_rate': 0.01, 'model__max_depth': 4, 'model__min_child_weight': 2, 'model__n_estimators': 2000, 'model__subsample': 0.8}
Best RMSE score: 25549.693098958334
Best MSE score: 671875253.3333334
Best R^2 score: 0.8873778025309245
---------------------------------------------------
Model saved in: saved_models\XGB_GS_Model.pkl


## Randomized grid search

In [9]:
params_distributions = {
            'model__n_estimators': np.arange(100, 2001, 200),
            'model__learning_rate': [0.01, 0.3, 0.05, 0.07, 0.1, 0.15],
            'model__max_depth': np.arange(1, 9),
            'model__min_child_weight': np.arange(1, 9),
            'model__subsample': [0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
            'model__colsample_bytree': [0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
            'model__reg_alpha': [0, 0.01, 0.1, 0.4, 0.6, 0.8, 1.0, 5.0],
            'model__reg_lambda': [0, 0.01, 0.1, 0.4, 0.6, 0.8, 1.0, 5.0, 10.0],
        }

random_grid = reporter.run_randomized_search(cv=5, params_distributions=params_distributions)

Start randomized grid search...
Randomized search finished!
Best params: {'model__subsample': 0.8, 'model__reg_lambda': 5.0, 'model__reg_alpha': 0.6, 'model__n_estimators': np.int64(1300), 'model__min_child_weight': np.int64(6), 'model__max_depth': np.int64(3), 'model__learning_rate': 0.05, 'model__colsample_bytree': 0.6}
Best RMSE score: 26026.126953125
Best MSE score: 693787667.2
Best R^2 score: 0.8837907711664835
---------------------------------------------------
Model saved in: saved_models\XGB_RGS_Model.pkl


## Zapis zbioru testowego do pliku

In [10]:
reporter.save_test_set()

Test set saved in: saved_models\XGB_test_set.csv


# LinearRegression

In [11]:
from sklearn.linear_model import LinearRegression

model = CustomRegressionModel(LinearRegression())
reporter = TrainingReporter(model, X_train, X_test, y_train, y_test, 'LR')

In [12]:
reporter.run_cross_validation(cv=5)

Start cross validation...
Iteration 1
Fold 0: RMSE = 34065.02831516518, MSE = 1160426154.1130056, R^2 = 0.8204733561260131
Fold 1: RMSE = 36348.06198601878, MSE = 1321181610.1394637, R^2 = 0.7742850723260825
Fold 2: RMSE = 41900.473418696965, MSE = 1755649672.710931, R^2 = 0.7393203022931261
Fold 3: RMSE = 29072.14767772229, MSE = 845189770.5952935, R^2 = 0.8451463134678403
Fold 4: RMSE = 33228.25368574767, MSE = 1104116843.004404, R^2 = 0.7907018256370844
Iteration 2
Fold 0: RMSE = 45053.91569253708, MSE = 2029855319.2302387, R^2 = 0.6933113243596751
Fold 1: RMSE = 32957.52650568964, MSE = 1086198553.3732352, R^2 = 0.8299836507424221
Fold 2: RMSE = 29199.661529034936, MSE = 852620233.4102029, R^2 = 0.8284196872696021
Fold 3: RMSE = 32512.90159166153, MSE = 1057088769.909067, R^2 = 0.7869465406068801
Fold 4: RMSE = 31145.66599999879, MSE = 970052510.5834806, R^2 = 0.8562346832267451
Iteration 3
Fold 0: RMSE = 25464.216145679293, MSE = 648426303.9138739, R^2 = 0.840047403133024
Fold 1: 

In [13]:
reporter.save_test_set()

Test set saved in: saved_models\LR_test_set.csv


# Random Forest

In [14]:
from sklearn.ensemble import RandomForestRegressor

model = CustomRegressionModel(RandomForestRegressor(
    n_estimators=1500,
    max_depth=10,
    min_samples_split=2,
    min_samples_leaf=2,
    max_features=0.3,
    random_state=42,
    n_jobs=-1
))

reporter = TrainingReporter(model, X_train, X_test, y_train, y_test, 'RF')
reporter.run_cross_validation(cv=5)

Start cross validation...
Iteration 1
Fold 0: RMSE = 28497.611533759682, MSE = 812113863.1290728, R^2 = 0.874359884276701
Fold 1: RMSE = 35036.738774950885, MSE = 1227573063.984147, R^2 = 0.7902774582804082
Fold 2: RMSE = 35031.93167036197, MSE = 1227236236.5569098, R^2 = 0.8177793804007667
Fold 3: RMSE = 24934.583149000435, MSE = 621733436.8144164, R^2 = 0.8860874586032784
Fold 4: RMSE = 24904.424611121816, MSE = 620230365.21105, R^2 = 0.882428128919867
Iteration 2
Fold 0: RMSE = 35818.748486891156, MSE = 1282982743.1671677, R^2 = 0.806155505447283
Fold 1: RMSE = 29269.554304947298, MSE = 856706809.210259, R^2 = 0.8659046602173227
Fold 2: RMSE = 25050.23476497558, MSE = 627514261.7803911, R^2 = 0.8737197534611358
Fold 3: RMSE = 29226.658684348156, MSE = 854197577.8513834, R^2 = 0.827838726370997
Fold 4: RMSE = 29622.404524148144, MSE = 877486849.7922724, R^2 = 0.8699532514493759
Iteration 3
Fold 0: RMSE = 23057.42237335823, MSE = 531644726.5034407, R^2 = 0.8688548658474016
Fold 1: RMS

In [15]:
reporter.save_test_set()

Test set saved in: saved_models\RF_test_set.csv
